# Random Impact Rule の外的再現（OpenAlex）

対象論文: Sinatra, Wang, Deville, Song, & Barabási, *Quantifying the evolution of individual scientific impact*, **Science** 354, aaf5239 (2016), [DOI](https://doi.org/10.1126/science.aaf5239)。

このノートブックは、既存の OpenAlex 応答キャッシュから、論文の Random Impact Rule の中心命題を再計算する。すなわち、各研究者の最高インパクト論文のキャリア年齢 $t^*$ が、各人の実際の年別出版量を条件にしたランダム配置と整合するかを検査する。

ここでの目的は元論文の完全な追試ではない。元論文とデータ源・著者同定・サンプル・分野・$c_{10}$ の実装が異なるため、**OpenAlexを用いる外的再現**として扱う。Q パラメータの推定は行わない。

## 固定入力と再現性

既存の `data/cache/openalex_random_impact_rule.json`（2026-06-13に保存されたOpenAlex API応答）を固定入力にする。外部APIを再照会しないため、API更新や引用数の後日変化に左右されず、同じ入力から同じ結果を検証できる。

既存実装の条件をそのまま引き継ぐ。Physics concept を含む論文から候補著者を抽出し、記事、出版年2015年以前、キャリア長20–60年、適格論文10本以上を満たす500名を対象とする。したがって「物理学者の代表標本」ではなく、**physics-seeded OpenAlex sample** と呼ぶ。既存キャッシュの選定には地域に関する暫定的な除外規則が含まれており、一般化可能な本分析では採用すべきではない。ここでは既存結果の再計算という目的に限って保持する。

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import html
import importlib
import json
import random
import sys

import pandas as pd

# ノートブックをリポジトリ内の任意の作業ディレクトリから実行できるようにする。
ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'openalex_random_impact_rule.py').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('リポジトリ直下を特定できません')
    ROOT = ROOT.parent

CACHE_PATH = ROOT / 'data' / 'cache' / 'openalex_random_impact_rule.json'
OUT_DIR = ROOT / 'data' / 'derived' / 'random_impact_rule'
FIGURE_PATH = ROOT / 'figures' / 'random_impact_rule_external.svg'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)

assert CACHE_PATH.exists(), f'固定入力が見つかりません: {CACHE_PATH}'

# 既存モジュールは requests を内部で利用するが、このノートブックでは下の OfflineClient により
# ネットワーク要求を禁止する。API取得済み応答だけを読むため、今回の再計算は標準ライブラリと pandas で完結する。
sys.path.insert(0, str(ROOT / 'src'))
import openalex_random_impact_rule as rir
importlib.reload(rir)

print(f'root: {ROOT}')
print(f'cache: {CACHE_PATH.name} ({CACHE_PATH.stat().st_size / 1024**2:.1f} MiB)')

root: /Users/jinedinujidan/Library/CloudStorage/GoogleDrive-r.ikura@msp-lab.org/My Drive/others/TA:RA/Sci-sci
cache: openalex_random_impact_rule.json (143.3 MiB)


In [2]:
class OfflineClient(rir.OpenAlexClient):
    """既存キャッシュにないAPI要求を失敗させるクライアント。"""

    def get(self, path, params=None, retries=5):
        params = dict(params or {})
        key = self._key(path, params)
        if key not in self.cache:
            raise RuntimeError(f'cache miss（ライブAPIは使わない）: {key}')
        return self.cache[key]

    def save_cache(self):
        # 固定入力を変更しない。
        return None


client = OfflineClient(cache_path=str(CACHE_PATH), sleep=0)
results = rir.run(
    client,
    field_filter='concepts.id:C121332964',  # キャッシュ作成時の旧Concept ID。現行の新規取得ではTopicsへ移行する。
    sample_works=50_000,
    candidate_limit=1_400,
    n_researchers=500,
    cutoff_year=2015,
    c10_window=10,
    min_career_years=20,
    max_career_years=60,
    min_papers=10,
    exclude_countries=rir.EXCLUDE_COUNTRIES,  # 固定入力を作った既存条件の再現に限る。
    repeats=500,
    seed=42,
    verbose=False,
)
records = results['records']
print(f"対象研究者: {len(records)}名 / 評価候補: {results['stats']['n_evaluated']}名")
print(f"既存実装の記述統計: KS={results['stats']['ks']:.4f}, Pearson r={results['stats']['pearson']:.4f}")

対象研究者: 500名 / 評価候補: 1074名
既存実装の記述統計: KS=0.1326, Pearson r=0.7512


## 検定の定義

研究者 $i$ の年別適格論文数を $n_i(t)$、適格論文総数を $N_i$ とする。帰無仮説では、最高インパクト論文の年齢はその研究者の全論文から一様に選ばれる。よって、集団の期待分布は

$$P_0(t)=\frac{1}{M}\sum_{i=1}^{M}\frac{n_i(t)}{N_i}$$

である。実分布 $P_{obs}(t)$ と $P_0(t)$ の累積分布の最大差を $D$ とし、同じ年別出版量を保って各研究者の最高論文位置をランダムに引き直す条件付きモンテカルロ検定で $p$ 値を求める。既存実装の500回シャッフル近似ではなく、ここでは $P_0(t)$ を解析的に計算する。

In [3]:
def pearson(xs, ys):
    """標準ライブラリだけでPearson相関を計算する。"""
    mx = sum(xs) / len(xs)
    my = sum(ys) / len(ys)
    cov = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    vx = sum((x - mx) ** 2 for x in xs)
    vy = sum((y - my) ** 2 for y in ys)
    return cov / (vx * vy) ** 0.5


def cdf_ks(a, b):
    """離散分布a,bの累積差の最大値を返す。"""
    ca = cb = d = 0.0
    for x, y in zip(a, b):
        ca += x
        cb += y
        d = max(d, abs(ca - cb))
    return d


max_t = max(r['span'] for r in records)
n_researchers = len(records)
p_observed = [0.0] * (max_t + 1)
p_null = [0.0] * (max_t + 1)

# 各研究者を同じ重みで1票とし、その人の論文数に比例する帰無分布を足し合わせる。
for record in records:
    p_observed[record['t_star']] += 1 / n_researchers
    n_eligible = sum(record['hist'].values())
    for publication_year, count in record['hist'].items():
        career_age = publication_year - record['first_year']
        p_null[career_age] += (count / n_eligible) / n_researchers

cdf_observed = []
cdf_null = []
running_observed = running_null = 0.0
for observed, expected in zip(p_observed, p_null):
    running_observed += observed
    running_null += expected
    cdf_observed.append(running_observed)
    cdf_null.append(running_null)

d_observed = cdf_ks(p_observed, p_null)
mean_t_observed = sum(t * p for t, p in enumerate(p_observed))
mean_t_null = sum(t * p for t, p in enumerate(p_null))
correlation = pearson(p_observed, p_null)

distribution_df = pd.DataFrame({
    'career_age_t': range(max_t + 1),
    'p_observed': p_observed,
    'p_conditional_null': p_null,
    'cdf_observed': cdf_observed,
    'cdf_conditional_null': cdf_null,
    'cdf_gap': [o - e for o, e in zip(cdf_observed, cdf_null)],
})
distribution_df.head()

,career_age_t,p_observed,p_conditional_null,cdf_observed,cdf_conditional_null,cdf_gap
0,0,0.014,0.025299,0.014,0.025299,-0.011299
1,1,0.004,0.014129,0.018,0.039428,-0.021428
2,2,0.016,0.014491,0.034,0.053918,-0.019918
3,3,0.018,0.018987,0.052,0.072905,-0.020905
4,4,0.014,0.021007,0.066,0.093912,-0.027912


In [4]:
def sample_null_distribution(records, max_t, rng):
    """
    各研究者について実際の年別出版量を重みとして1本を抽選する。
    これにより、個人ごとの生産性とキャリア長を保ったままRandom Impact Ruleの帰無標本を作る。
    """
    counts = [0] * (max_t + 1)
    for record in records:
        ages = [year - record['first_year'] for year in record['hist']]
        weights = [record['hist'][year] for year in record['hist']]
        selected_age = rng.choices(ages, weights=weights, k=1)[0]
        counts[selected_age] += 1
    return [count / len(records) for count in counts]


B = 10_000
rng = random.Random(20_260_716)
null_statistics = []
for _ in range(B):
    simulated = sample_null_distribution(records, max_t, rng)
    null_statistics.append(cdf_ks(simulated, p_null))

null_statistics.sort()
n_extreme = sum(value >= d_observed for value in null_statistics)
p_value_mc = (n_extreme + 1) / (B + 1)  # 有限回数のモンテカルロ検定の補正

summary = {
    'source_paper_doi': '10.1126/science.aaf5239',
    'analysis_type': 'external replication of the random-impact-rule timing test',
    'n_researchers': n_researchers,
    'n_candidates_evaluated': results['stats']['n_evaluated'],
    'max_career_age': max_t,
    'ks_distance_observed_vs_conditional_null': d_observed,
    'pearson_observed_vs_conditional_null': correlation,
    'mean_t_star_observed': mean_t_observed,
    'mean_t_star_conditional_null': mean_t_null,
    'mean_difference_years': mean_t_observed - mean_t_null,
    'monte_carlo_repetitions': B,
    'monte_carlo_p_value': p_value_mc,
    'null_ks_mean': sum(null_statistics) / B,
    'null_ks_95pct': null_statistics[int((B - 1) * 0.95)],
    'null_ks_99pct': null_statistics[int((B - 1) * 0.99)],
    'seed': 20_260_716,
    'cutoff_year': 2015,
    'c10_window_years': 10,
}
pd.DataFrame([summary]).T.rename(columns={0: 'value'}).round(6)

,value
source_paper_doi,10.1126/science.aaf5239
analysis_type,external replication of the random-impact-rule...
n_researchers,500
n_candidates_evaluated,1074
max_career_age,60
ks_distance_observed_vs_conditional_null,0.133106
pearson_observed_vs_conditional_null,0.752737
mean_t_star_observed,24.884
mean_t_star_conditional_null,21.280469
mean_difference_years,3.603531


In [5]:
# 研究者ごとの入力を保存する。年別ヒストグラムも残すため、帰無分布を後から再計算できる。
record_rows = []
for record in records:
    row = {key: value for key, value in record.items() if key != 'hist'}
    row['publication_year_histogram_json'] = json.dumps(
        dict(sorted(record['hist'].items())), ensure_ascii=False, sort_keys=True
    )
    record_rows.append(row)

records_path = OUT_DIR / 'records.csv'
distribution_path = OUT_DIR / 'distribution.csv'
summary_path = OUT_DIR / 'summary.json'

pd.DataFrame(record_rows).to_csv(records_path, index=False)
distribution_df.to_csv(distribution_path, index=False)

def file_digest(path):
    return sha256(path.read_bytes()).hexdigest()

manifest = {
    'analysis_executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_cache': {
        'path': str(CACHE_PATH.relative_to(ROOT)),
        'sha256': file_digest(CACHE_PATH),
        'modified_at_utc': datetime.fromtimestamp(CACHE_PATH.stat().st_mtime, timezone.utc).isoformat(),
    },
    'parameters': {
        'field_filter_at_cache_creation': 'concepts.id:C121332964',
        'sample_works': 50_000,
        'candidate_limit': 1_400,
        'n_researchers': 500,
        'type_filter': 'article',
        'cutoff_year': 2015,
        'c10_window': 10,
        'career_span_years': [20, 60],
        'minimum_eligible_papers': 10,
    },
    'summary': summary,
    'outputs': {
        'records.csv_sha256': file_digest(records_path),
        'distribution.csv_sha256': file_digest(distribution_path),
    },
}
summary_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'saved: {records_path.relative_to(ROOT)}')
print(f'saved: {distribution_path.relative_to(ROOT)}')
print(f'saved: {summary_path.relative_to(ROOT)}')

saved: data/derived/random_impact_rule/records.csv
saved: data/derived/random_impact_rule/distribution.csv
saved: data/derived/random_impact_rule/summary.json


In [6]:
def svg_polyline(values, x0, y0, width, height, ymax):
    points = []
    denominator = max(1, len(values) - 1)
    for index, value in enumerate(values):
        x = x0 + width * index / denominator
        y = y0 + height * (1 - value / ymax)
        points.append(f'{x:.2f},{y:.2f}')
    return ' '.join(points)


# matplotlib を使わず、標準ライブラリで再現結果を閲覧できるSVGにする。
svg_width, svg_height = 920, 500
left, top, plot_width, plot_height = 80, 70, 770, 320
ymax = max(max(p_observed), max(p_null)) * 1.15
observed_line = svg_polyline(p_observed, left, top, plot_width, plot_height, ymax)
null_line = svg_polyline(p_null, left, top, plot_width, plot_height, ymax)
ticks = []
for t in range(0, max_t + 1, 10):
    x = left + plot_width * t / max_t
    ticks.append(f'<line x1="{x:.2f}" y1="{top + plot_height}" x2="{x:.2f}" y2="{top + plot_height + 6}" stroke="#333"/>')
    ticks.append(f'<text x="{x:.2f}" y="{top + plot_height + 25}" text-anchor="middle" font-size="12">{t}</text>')
for fraction in (0, 0.25, 0.5, 0.75, 1):
    y = top + plot_height * (1 - fraction)
    label = f'{ymax * fraction:.3f}'
    ticks.append(f'<line x1="{left - 6}" y1="{y:.2f}" x2="{left}" y2="{y:.2f}" stroke="#333"/>')
    ticks.append(f'<line x1="{left}" y1="{y:.2f}" x2="{left + plot_width}" y2="{y:.2f}" stroke="#e5e7eb"/>')
    ticks.append(f'<text x="{left - 10}" y="{y + 4:.2f}" text-anchor="end" font-size="12">{label}</text>')

title = html.escape('Random Impact Rule: observed timing vs conditional null')
subtitle = html.escape(f'N={n_researchers}; D={d_observed:.3f}; conditional Monte Carlo p={p_value_mc:.4f}')
svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="{svg_width}" height="{svg_height}" viewBox="0 0 {svg_width} {svg_height}">
  <rect width="100%" height="100%" fill="white"/>
  <text x="{svg_width / 2}" y="30" text-anchor="middle" font-family="sans-serif" font-size="19">{title}</text>
  <text x="{svg_width / 2}" y="52" text-anchor="middle" font-family="sans-serif" font-size="13">{subtitle}</text>
  <line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_height}" stroke="#333"/>
  <line x1="{left}" y1="{top + plot_height}" x2="{left + plot_width}" y2="{top + plot_height}" stroke="#333"/>
  {''.join(ticks)}
  <polyline fill="none" stroke="#b91c1c" stroke-width="2.4" points="{observed_line}"/>
  <polyline fill="none" stroke="#1e3a5f" stroke-width="2.4" stroke-dasharray="7 4" points="{null_line}"/>
  <line x1="610" y1="420" x2="642" y2="420" stroke="#b91c1c" stroke-width="2.4"/>
  <text x="650" y="424" font-family="sans-serif" font-size="13">Observed</text>
  <line x1="720" y1="420" x2="752" y2="420" stroke="#1e3a5f" stroke-width="2.4" stroke-dasharray="7 4"/>
  <text x="760" y="424" font-family="sans-serif" font-size="13">Conditional null</text>
  <text x="{left + plot_width / 2}" y="{top + plot_height + 55}" text-anchor="middle" font-family="sans-serif" font-size="14">Career age of highest-impact paper, t*</text>
  <text x="22" y="{top + plot_height / 2}" text-anchor="middle" transform="rotate(-90 22 {top + plot_height / 2})" font-family="sans-serif" font-size="14">Probability</text>
</svg>'''
FIGURE_PATH.write_text(svg, encoding='utf-8')
print(f'saved: {FIGURE_PATH.relative_to(ROOT)}')

saved: figures/random_impact_rule_external.svg


## 解釈

この固定スナップショットでは、観測分布と条件付き帰無分布の差は、帰無モデルからの典型的なばらつきより大きい。したがって、この標本だけでは Random Impact Rule を支持しない。最高インパクト論文は、実際の年別出版量で説明される位置より平均して後ろに現れる。

これは元論文への反証ではない。理由は、(1) OpenAlex と元論文のデータが異なる、(2) 本分析は個々の論文順ではなく年単位で位置を近似する、(3) `c10` は既存APIキャッシュの被引用リンクから求めた代理指標である、(4) 最高論文候補を総被引用数上位20本に制限している、(5) 著者名寄せ・分野抽出・右側打ち切りの影響がある、ためである。

次のライブ再現では、現行OpenAlexのTopicsを用い、ORCID/CVで検証した著者群を事前に固定し、全適格論文の10年引用を取得し、論文順（同一年内の順序も含む）で検査する。地域・国籍・文字種による除外は用いず、不確実性フラグと感度分析で名寄せ誤差を扱う。

In [7]:
assert len(records) == 500, '研究者数が固定再現条件と一致しません'
assert abs(sum(p_observed) - 1.0) < 1e-12, '観測分布が正規化されていません'
assert abs(sum(p_null) - 1.0) < 1e-12, '帰無分布が正規化されていません'
assert all(sum(r['hist'].values()) == r['n_eligible'] for r in records), '年別論文数が不整合です'

for path in (records_path, distribution_path, summary_path, FIGURE_PATH):
    assert path.exists() and path.stat().st_size > 0, f'出力検証に失敗: {path}'
    print(f'OK  {path.relative_to(ROOT)}  {path.stat().st_size:,} bytes')

manifest = json.loads(summary_path.read_text(encoding='utf-8'))
manifest['outputs']['random_impact_rule_external.svg_sha256'] = file_digest(FIGURE_PATH)
summary_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print('再現計算と出力検証が完了しました。')

OK  data/derived/random_impact_rule/records.csv  266,145 bytes
OK  data/derived/random_impact_rule/distribution.csv  5,877 bytes
OK  data/derived/random_impact_rule/summary.json  1,650 bytes
OK  figures/random_impact_rule_external.svg  4,946 bytes
再現計算と出力検証が完了しました。
